In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Load the dataset
df = pd.read_csv('../tabela-fipe-329.csv')

# Display head and info
print(df.head())
print(df.info())
print(df['Brand_Code'].unique())

  Type  Brand_Code Brand_Value  Model_Code     Model_value Year_Code  \
0  CAR           1       Acura           1  Integra GS 1.8    1992-1   
1  CAR           1       Acura           2  Legend 3.2/3.5    1996-1   
2  CAR           1       Acura           2  Legend 3.2/3.5    1991-1   
3  CAR           1       Acura           2  Legend 3.2/3.5    1995-1   
4  CAR           1       Acura           1  Integra GS 1.8    1991-1   

      Year_Value Fipe_Code Fuel_Letter Fuel_Type         Price  \
0  1992 Gasolina  038003-2           G  Gasolina  R$ 10.942,00   
1  1996 Gasolina  038002-4           G  Gasolina  R$ 20.941,00   
2  1991 Gasolina  038002-4           G  Gasolina  R$ 14.022,00   
3  1995 Gasolina  038002-4           G  Gasolina  R$ 18.821,00   
4  1991 Gasolina  038003-2           G  Gasolina  R$ 10.221,00   

             Month  
0  janeiro de 2026  
1  janeiro de 2026  
2  janeiro de 2026  
3  janeiro de 2026  
4  janeiro de 2026  
<class 'pandas.core.frame.DataFrame'>
RangeI

In [3]:
# List of brand IDs to filter
brand_ids = [6, 238, 23, 13, 21, 22, 25, 26, 208, 177, 29, 31, 39, 41, 43, 44, 48, 56, 59, 57, 58]

# Filter the dataframe
df_filtered = df[df['Brand_Code'].isin(brand_ids)].copy()

# Clean Price column
# Remove 'R$ ', '.', and replace ',' with '.'
df_filtered['Price_Numeric'] = df_filtered['Price'].astype(str).str.replace('R$ ', '', regex=False).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
df_filtered['Price_Numeric'] = pd.to_numeric(df_filtered['Price_Numeric'], errors='coerce')

# Clean Year column
# Extract the year from 'Year_Value'. Handle '32000' or similar if present (sometimes 32000 is used for Zero KM in FIPE, but let's check).
# Usually 'Year_Value' is 'YYYY Fuel'.
# Let's check unique values of Year_Value to be sure.
print("Unique Year_Value samples:", df_filtered['Year_Value'].unique()[:20])

# Extract year (first 4 digits)
df_filtered['Year_Numeric'] = df_filtered['Year_Value'].astype(str).str.extract(r'(\d{4})').astype(float)

# Handle 32000 (Zero KM) - usually FIPE represents 32000 as Zero KM.
# I will replace 32000 with the current year (2026) or 2025/2026 depending on context, or keep it as a separate category?
# The prompt says "Previsão de Preços... Janeiro de 2026".
# Let's see if 32000 exists.
print("Count of Year 32000:", (df_filtered['Year_Numeric'] == 32000).sum())

# If 32000 exists, it means Zero KM. Since the data is Jan 2026, Zero KM is likely 2026 or 2025 model.
# I will replace 32000 with 2026 for numeric analysis, or maybe create a 'Is_ZeroKM' flag.
# Let's replace with 2026 for now, as it's a numeric year column.
df_filtered.loc[df_filtered['Year_Numeric'] == 32000, 'Year_Numeric'] = 2026

# Drop rows with NaN in Price or Year
df_filtered.dropna(subset=['Price_Numeric', 'Year_Numeric'], inplace=True)

# Basic Stats
print(df_filtered[['Price_Numeric', 'Year_Numeric']].describe())

# Check brands present
print("Brands present:", df_filtered['Brand_Value'].unique())

Unique Year_Value samples: ['1995 Gasolina' '1994 Gasolina' '1993 Gasolina' '1996 Gasolina'
 '1999 Gasolina' '1997 Gasolina' '1998 Gasolina' '2011 Gasolina'
 '2012 Gasolina' '2014 Gasolina' '2013 Gasolina' '2016 Gasolina'
 '2017 Gasolina' '2015 Gasolina' '2018 Gasolina' '2003 Gasolina'
 '2000 Gasolina' '2002 Gasolina' '2001 Gasolina' '2005 Gasolina']
Count of Year 32000: 0
       Price_Numeric  Year_Numeric
count   2.280100e+04  22801.000000
mean    8.584784e+04   2042.372308
std     1.358562e+05    197.141988
min     2.023000e+03   1985.000000
25%     2.169500e+04   2001.000000
50%     4.585900e+04   2010.000000
75%     9.587600e+04   2018.000000
max     4.094675e+06   3200.000000
Brands present: ['Audi' 'BYD' 'Citroën' 'Fiat' 'Ford' 'GM - Chevrolet' 'Honda' 'Hyundai'
 'IVECO' 'JAC' 'Jeep' 'Kia Motors' 'Mercedes-Benz' 'Mitsubishi' 'Nissan'
 'Peugeot' 'Renault' 'Toyota' 'Troller' 'Volvo' 'VW - VolksWagen']


In [4]:
# Fix Year 3200 -> 2026
df_filtered.loc[df_filtered['Year_Numeric'] == 3200, 'Year_Numeric'] = 2026

# Check Type column
print("Unique Types:", df_filtered['Type'].unique())

# Generate plots
# 1. Price Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df_filtered['Price_Numeric'], bins=50, kde=True)
plt.title('Distribuição de Preços')
plt.xlabel('Preço (R$)')
plt.ylabel('Frequência')
plt.savefig('price_distribution.png')
plt.close()

# 2. Boxplot Price by Brand
plt.figure(figsize=(14, 8))
sns.boxplot(x='Brand_Value', y='Price_Numeric', data=df_filtered)
plt.xticks(rotation=45, ha='right')
plt.title('Distribuição de Preços por Fabricante')
plt.xlabel('Fabricante')
plt.ylabel('Preço (R$)')
plt.tight_layout()
plt.savefig('price_by_brand.png')
plt.close()

# 3. Correlation Matrix
# Select numeric columns
numeric_cols = ['Price_Numeric', 'Year_Numeric']
corr_matrix = df_filtered[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação')
plt.savefig('correlation_matrix.png')
plt.close()

# 4. Scatter Plot: Price vs Year
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Year_Numeric', y='Price_Numeric', data=df_filtered, alpha=0.5)
plt.title('Preço vs Ano de Fabricação')
plt.xlabel('Ano')
plt.ylabel('Preço (R$)')
plt.savefig('price_vs_year.png')
plt.close()

# 5. Bar Plot: Count by Brand
plt.figure(figsize=(14, 8))
sns.countplot(x='Brand_Value', data=df_filtered, order=df_filtered['Brand_Value'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.title('Contagem de Veículos por Fabricante')
plt.xlabel('Fabricante')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.savefig('count_by_brand.png')
plt.close()

# 6. Bar Plot: Avg Price by Brand
avg_price_brand = df_filtered.groupby('Brand_Value')['Price_Numeric'].mean().sort_values(ascending=False)
plt.figure(figsize=(14, 8))
sns.barplot(x=avg_price_brand.index, y=avg_price_brand.values)
plt.xticks(rotation=45, ha='right')
plt.title('Preço Médio por Fabricante')
plt.xlabel('Fabricante')
plt.ylabel('Preço Médio (R$)')
plt.tight_layout()
plt.savefig('avg_price_by_brand.png')
plt.close()

# 7. Boxplot: Price by Fuel Type
plt.figure(figsize=(10, 6))
sns.boxplot(x='Fuel_Type', y='Price_Numeric', data=df_filtered)
plt.xticks(rotation=45, ha='right')
plt.title('Preço por Tipo de Combustível')
plt.xlabel('Combustível')
plt.ylabel('Preço (R$)')
plt.tight_layout()
plt.savefig('price_by_fuel.png')
plt.close()

# Save cleaned data for user reference (optional but good practice)
# df_filtered.to_csv('fipe_jan2026_cleaned.csv', index=False)
# Since user cannot access file unless I mention it, I will just describe the data.

print("EDA completed. Plots saved.")
print(df_filtered.groupby('Brand_Value')['Price_Numeric'].describe())

Unique Types: ['CAR']
EDA completed. Plots saved.
                  count           mean            std       min        25%  \
Brand_Value                                                                  
Audi             1166.0  182645.974271  215436.662793   10525.0   31885.75   
BYD                59.0  210990.440678   87694.914967   99160.0  161832.00   
Citroën           856.0   47396.716121   47539.847966    2767.0   12745.50   
Fiat             2481.0   45709.512696   46658.450681    2023.0   15345.00   
Ford             2253.0   64480.652463   82127.601574    2472.0   15522.00   
GM - Chevrolet   2599.0   57787.630627   70420.484062    3069.0   19038.50   
Honda             603.0   68750.456053   56669.767595    6121.0   26885.00   
Hyundai           806.0   63599.413151   48032.816149    2906.0   33444.00   
IVECO             210.0  256481.419048   73908.154158  116927.0  191798.25   
JAC               140.0  112307.507143  106758.374281   14032.0   32756.00   
Jeep          